In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric
from sklearn.metrics import roc_auc_score



In [3]:
data = pd.read_csv(r'C:\Users\Anton\Downloads\archive (1)\creditcard.csv')
data = data.drop(['Time'], axis=1)

y = data['Class']
X = data.drop(['Class'], axis=1)

# scaling the amount to the other features
X['Amount'] = np.log(X['Amount'] + 0.001)
scaler = RobustScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

X

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0.730845
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.801054
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,1.084841
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0.657702
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0.441202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,...,1.475829,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,-1.277597
284803,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,...,0.059616,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,0.045517
284804,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,...,0.001396,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,0.429532
284805,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,...,0.127434,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,-0.300571


In [4]:
model_path = 'bgm_model_antoni.joblib'

if os.path.exists(model_path):
    print("Loading saved model...")
    bgm = joblib.load(model_path)
else:
    bgm = BayesianGaussianMixture(
        n_components=15,
        weight_concentration_prior_type='dirichlet_process',
        weight_concentration_prior=1e-3,
        max_iter=500,
        random_state=42,
        verbose=1
    )
    bgm.fit(X)
    joblib.dump(bgm, model_path)
    print(f"Model saved locally to {model_path}")

Loading saved model...


In [5]:
weights = np.round(bgm.weights_, 4)
active_clusters = np.where(weights > 0.005)[0]

print(f"active clusters: {weights[active_clusters]}")


active clusters: [0.1173 0.0324 0.0513 0.0137 0.3501 0.0557 0.1214 0.007  0.0071 0.0532
 0.0163 0.0096 0.0803 0.045  0.0396]


In [6]:
results = pd.DataFrame()
results['Cluster'] = bgm.predict(X)
results['True_Class'] = y.values 

crosstab = pd.crosstab(results['Cluster'], results['True_Class'])

crosstab = crosstab.rename(columns={0: 'Normal', 1: 'Fraud'})

if 'Fraud' not in crosstab.columns: crosstab['Fraud'] = 0
if 'Normal' not in crosstab.columns: crosstab['Normal'] = 0

crosstab['Fraud_Rate (%)'] = (crosstab['Fraud'] / (crosstab['Normal'] + crosstab['Fraud'])) * 100

print(crosstab.sort_values(by='Fraud_Rate (%)', ascending=False))

True_Class  Normal  Fraud  Fraud_Rate (%)
Cluster                                  
8             1675    346       17.120238
7             1942     61        3.045432
3             3856      7        0.181206
11            2718      4        0.146951
2            14592     10        0.068484
10            4619      3        0.064907
4            99669     39        0.039114
0            33313     11        0.033009
9            15165      5        0.032960
12           22927      4        0.017444
13           12828      1        0.007795
5            15851      1        0.006308
1             9232      0        0.000000
6            34648      0        0.000000
14           11280      0        0.000000


In [7]:
#  We separate 2 good clusters from the rest of the data
best_two_clusters = results[results['Cluster'].isin([7, 8])]
X_best_two_clusters = X.iloc[best_two_clusters.index]
y_best_two_clusters = y.iloc[best_two_clusters.index]

X_worst = X[~X.index.isin(best_two_clusters.index)]
y_worst = y[~y.index.isin(best_two_clusters.index)]

In [25]:
data_worst = pd.concat([X_worst, y_worst], axis=1)
data_worst

worst1 = data_worst[data_worst['Class'] == 1]
worst0 = data_worst[data_worst['Class'] == 0]

# fraudulent data
X_worst1 = worst1.drop(['Class'], axis=1)
y_worst1 = worst1['Class']

# not fraudulent data
X_worst0 = worst0.drop(['Class'], axis=1)
y_worst0 = worst0['Class']

X_worst0_train, X_worst0_test, y_worst0_train, y_worst0_test = train_test_split(X_worst0, y_worst0, test_size=0.2, random_state=42)

X_worst_train, X_temp, y_worst_train, y_temp = train_test_split(
    X_worst, y_worst, test_size=0.2, random_state=42, stratify=y_worst
)

X_worst_validate, X_worst_test, y_worst_validate, y_worst_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


### TabNet trained on the shit clusters 

In [26]:
clf = TabNetClassifier()  #TabNetRegressor()
clf.fit(
  X_worst_train.values, y_worst_train.values,
  eval_set=[(X_worst_validate.values, y_worst_validate.values)],
  eval_metric=['auc'],
  weights=1
)
preds = clf.predict(X_worst_test.values)

y_true = y_worst_test.values.ravel()
cm = confusion_matrix(y_true, preds)

print("--- Confusion Matrix ---")
print(f"True Negatives (Normal as Normal): {cm[0][0]}")
print(f"False Positives (Normal as Fraud): {cm[0][1]}")
print(f"False Negatives (Fraud as Normal): {cm[1][0]}")
print(f"True Positives  (Fraud as Fraud):  {cm[1][1]}\n")

print("--- Classification Report ---")
print(classification_report(y_true, preds))


c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.22789 | val_0_auc: 0.8606  |  0:00:28s
epoch 1  | loss: 0.08307 | val_0_auc: 0.83224 |  0:01:52s
epoch 2  | loss: 0.05891 | val_0_auc: 0.87654 |  0:02:56s
epoch 3  | loss: 0.0459  | val_0_auc: 0.81926 |  0:03:22s
epoch 4  | loss: 0.04309 | val_0_auc: 0.85741 |  0:03:45s
epoch 5  | loss: 0.04115 | val_0_auc: 0.78795 |  0:04:06s
epoch 6  | loss: 0.03168 | val_0_auc: 0.84205 |  0:04:28s
epoch 7  | loss: 0.02641 | val_0_auc: 0.83294 |  0:04:50s
epoch 8  | loss: 0.03436 | val_0_auc: 0.81594 |  0:05:12s
epoch 9  | loss: 0.0295  | val_0_auc: 0.84725 |  0:05:34s
epoch 10 | loss: 0.02754 | val_0_auc: 0.81679 |  0:05:56s
epoch 11 | loss: 0.03324 | val_0_auc: 0.78314 |  0:06:17s
epoch 12 | loss: 0.02979 | val_0_auc: 0.78393 |  0:06:40s

Early stopping occurred at epoch 12 with best_epoch = 2 and best_val_0_auc = 0.87654


c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


--- Confusion Matrix ---
True Negatives (Normal as Normal): 27645
False Positives (Normal as Fraud): 425
False Negatives (Fraud as Normal): 3
True Positives  (Fraud as Fraud):  6

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     28070
           1       0.01      0.67      0.03         9

    accuracy                           0.98     28079
   macro avg       0.51      0.83      0.51     28079
weighted avg       1.00      0.98      0.99     28079



Here we achieve 66% accuracy on fraudulent data with false positives being 1.51% of the test set. In comparison, one-class svm achieves 56% accuracy whith flase positives being 10.27% of the test set. We find the significant decrease in the number of false positives, while mainaining good accuracy to be the most crucial finding.

Due to small number of fraude in the test set, we ran the model on multiple random states to confirm these results and found them to be generally inconsistent. We will, therefore, move from randomised data splits to testing the model with all fraud cases by splitting fraud in into 9 batches.

This also brings us to an interesting hypothesis that different models might be good at predicting different types of camuflaged fraud. We will examine that by looking at specific fraud cases and how they are classified by different models. This analysis will be difficult to conduct as features are meaningless, but we will try nonetheless.

### TabNet trained on the whole dataset

In [27]:
X_train_whole, X_temp_whole, y_train_whole, y_temp_whole = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_validate_whole, X_test_whole, y_validate_whole, y_test_whole = train_test_split(X_temp_whole, y_temp_whole, test_size=0.5, random_state=42, stratify=y_temp_whole)

In [28]:
clf_big = TabNetClassifier()  #TabNetRegressor()
clf_big.fit(
  X_train_whole.values, y_train_whole.values,
  eval_set=[(X_validate_whole.values, y_validate_whole.values)],
  eval_metric=['auc'],
  weights=1
)
preds = clf_big.predict(X_test_whole.values)

y_true = y_test_whole.values.ravel()
cm = confusion_matrix(y_true, preds)

print("--- Confusion Matrix ---")
print(f"True Negatives (Normal as Normal): {cm[0][0]}")
print(f"False Positives (Normal as Fraud): {cm[0][1]}")
print(f"False Negatives (Fraud as Normal): {cm[1][0]}")
print(f"True Positives  (Fraud as Fraud):  {cm[1][1]}\n")

print("--- Classification Report ---")
print(classification_report(y_true, preds))

c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.16186 | val_0_auc: 0.95986 |  0:00:29s
epoch 1  | loss: 0.06842 | val_0_auc: 0.96624 |  0:00:53s
epoch 2  | loss: 0.05543 | val_0_auc: 0.95771 |  0:01:18s
epoch 3  | loss: 0.03901 | val_0_auc: 0.96354 |  0:01:43s
epoch 4  | loss: 0.0306  | val_0_auc: 0.95583 |  0:02:07s
epoch 5  | loss: 0.02919 | val_0_auc: 0.9498  |  0:02:34s
epoch 6  | loss: 0.02515 | val_0_auc: 0.94053 |  0:02:59s
epoch 7  | loss: 0.02378 | val_0_auc: 0.94496 |  0:03:24s
epoch 8  | loss: 0.02093 | val_0_auc: 0.94597 |  0:03:48s
epoch 9  | loss: 0.0199  | val_0_auc: 0.93909 |  0:04:14s
epoch 10 | loss: 0.01956 | val_0_auc: 0.95263 |  0:04:41s
epoch 11 | loss: 0.01853 | val_0_auc: 0.9569  |  0:05:09s

Early stopping occurred at epoch 11 with best_epoch = 1 and best_val_0_auc = 0.96624


c:\Users\Anton\AppData\Local\Programs\Python\Python313\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


--- Confusion Matrix ---
True Negatives (Normal as Normal): 27773
False Positives (Normal as Fraud): 659
False Negatives (Fraud as Normal): 3
True Positives  (Fraud as Fraud):  46

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     28432
           1       0.07      0.94      0.12        49

    accuracy                           0.98     28481
   macro avg       0.53      0.96      0.56     28481
weighted avg       1.00      0.98      0.99     28481



When we compare these results to the xgb we find that accuracy is similar but number of false positives is higher. This finding is not really relevant to anything, but we find it interesting as it tells us a little about the model architecture. 